# T 国债期货数据管线：Wind → DuckDB

Notebook 只保留运行参数、全量/增量开关和结果检查；实现集中在 `src/treasury_futures/data_pipeline.py`。


In [1]:
from pathlib import Path
import sys

_PROJECT_CANDIDATES = (Path.cwd(), *Path.cwd().parents)
PROJECT_ROOT = next(
    (candidate for candidate in _PROJECT_CANDIDATES if (candidate / "pyproject.toml").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("找不到项目根目录 pyproject.toml；请从本项目内启动 Jupyter。")
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from IPython.display import display
from treasury_futures.data_pipeline import *  # noqa: F403


## 6. 运行方式

下面两个单元格默认不执行 Wind 请求。把对应开关改为 `True` 后运行即可。首次建库先导入一次 `T_mindf.csv`，再执行全量流程；日常维护使用增量流程，不必重复导入 CSV。Wind 分钟函数无论传入多早的 `START_DATE`，都会自动跳过 `T2503.CFE` 以前的合约。


In [ ]:
# ------------------------------ 首次全量建库 ------------------------------
RUN_FULL_UPDATE = False

if RUN_FULL_UPDATE:
    from WindPy import w

    w.start(waitTime=60)
    if not w.isconnected():
        raise ConnectionError("WindPy 未连接")

    con = open_database(DB_PATH)

    # 历史 CSV 只需首次导入；分块、原子、可重复执行，且不会覆盖已有 Wind 主键。
    csv_import_result = import_historical_minute_csv(
        con, HISTORICAL_MINUTE_CSV
    )

    # 日线函数会先更新合约目录和主力映射。
    daily_result = full_update_daily(w, con, START_DATE, END_DATE)

    # Wind 分钟线仅下载 T2503.CFE 及之后的合约。
    minute_result = full_update_minute(
        w, con, START_DATE, END_DATE,
        refresh_reference=False,
        pause_seconds=0.0,
    )

    # 2024-11-21 的 CSV 主力为 T2503，但同键 Wind 行会挡住普通 INSERT OR IGNORE；显式覆盖该日。
    csv_gap_patch_result = patch_historical_minute_date_from_csv(
        con, "2024-11-21", HISTORICAL_MINUTE_CSV
    )

    continuous_result = build_main_continuous(con, START_DATE, END_DATE)
    adj_factor_list = build_adj_factor_list(con, strict=True)

    display(csv_import_result)
    display(daily_result.tail())
    display(minute_result.tail())
    display(csv_gap_patch_result)
    display(continuous_result)
    display(adj_factor_list.tail())


In [ ]:
# ------------------------------ 日常增量更新 ------------------------------
RUN_INCREMENTAL_UPDATE = False

if RUN_INCREMENTAL_UPDATE:
    from WindPy import w

    w.start(waitTime=60)
    if not w.isconnected():
        raise ConnectionError("WindPy 未连接")

    con = open_database(DB_PATH)
    update_end = datetime.now()
    # 若这是一个全新数据库，请先单独运行：
    # import_historical_minute_csv(con, HISTORICAL_MINUTE_CSV)

    daily_result = incremental_update_daily(
        w, con, bootstrap_start_date=START_DATE, end_date=update_end.date(),
        refresh_reference=True,
    )
    minute_result = incremental_update_minute(
        w, con, bootstrap_start_date=START_DATE, end_time=update_end,
        refresh_reference=False,
    )

    # 为简单可靠，这里重建整个连续区间；也可以传入较短 start/end 只重建近期。
    continuous_result = build_main_continuous(con)
    adj_factor_list = build_adj_factor_list(con, strict=True)

    display(daily_result.tail())
    display(minute_result.tail())
    display(continuous_result)
    display(adj_factor_list.tail())


## 7. DuckDB 检查与调用示例

`main_daily_continuous`、`main_minute_continuous` 是未复权数据；`v_main_daily_backward_adjusted`、`v_main_minute_backward_adjusted` 是应用结算价因子后的便捷视图。


In [ ]:
con = open_database(DB_PATH)

# 各表行数和覆盖区间
table_overview = con.execute(
    """
    SELECT 'contracts' AS table_name, count(*) AS rows, NULL AS min_time, NULL AS max_time FROM contracts
    UNION ALL
    SELECT 'main_contract_mapping', count(*), min(trade_date)::VARCHAR, max(trade_date)::VARCHAR FROM main_contract_mapping
    UNION ALL
    SELECT 'main_contract_rolls', count(*), min(roll_date)::VARCHAR, max(roll_date)::VARCHAR FROM main_contract_rolls
    UNION ALL
    SELECT 'daily_bars', count(*), min(trade_date)::VARCHAR, max(trade_date)::VARCHAR FROM daily_bars
    UNION ALL
    SELECT 'minute_bars', count(*), min(bar_time)::VARCHAR, max(bar_time)::VARCHAR FROM minute_bars
    UNION ALL
    SELECT 'main_daily_continuous', count(*), min(trade_date)::VARCHAR, max(trade_date)::VARCHAR FROM main_daily_continuous
    UNION ALL
    SELECT 'main_minute_continuous', count(*), min(bar_time)::VARCHAR, max(bar_time)::VARCHAR FROM main_minute_continuous
    UNION ALL
    SELECT 'main_adj_factor_list', count(*), min(segment_start)::VARCHAR, max(segment_end)::VARCHAR FROM main_adj_factor_list
    """
).fetchdf()
display(table_overview)

# 分钟数据来源覆盖情况
display(con.execute(
    """
    SELECT data_source, count(*) AS rows, min(bar_time) AS min_time,
           max(bar_time) AS max_time, count(DISTINCT wind_code) AS contracts
    FROM minute_bars GROUP BY data_source ORDER BY data_source
    """
).fetchdf())

# 最近的主力映射与连续数据
display(con.execute("SELECT * FROM main_contract_mapping ORDER BY trade_date DESC LIMIT 10").fetchdf())
display(con.execute("SELECT * FROM main_daily_continuous ORDER BY trade_date DESC LIMIT 10").fetchdf())
display(con.execute("SELECT * FROM main_adj_factor_list ORDER BY segment_start DESC LIMIT 10").fetchdf())

# 调用示例：
daily_df = con.execute("SELECT * FROM main_daily_continuous ORDER BY trade_date").fetchdf()
minute_df = con.execute("SELECT * FROM main_minute_continuous ORDER BY bar_time").fetchdf()
adjusted_daily_df = con.execute("SELECT * FROM v_main_daily_backward_adjusted ORDER BY trade_date").fetchdf()

# 结束运行
con.close()
